# Create focus-test Dave config

Builds the focus-lock test Dave recipe: visits every FOV and checks the
focus lock, with no fluidics, before the full acquisition. Needs only the
positions (02) and notebook 01's focus-test HAL config. Run its recipe on
the microscope first; then (tumor_disk) `02d_filter_positions_by_tissue_quality.ipynb`,
then 03 and 04 on the kept FOVs.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta
from IPython.display import display

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent  # MERci/ (notebook lives in MERci/notebooks/before_imaging/regular/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.dave   import (
    dave_focustest_config_filename, create_focus_test_dave_config, experiment_estimate_table,
)
from MERci.acquisition.kilroy import find_kilroy_config
from MERci.acquisition.display import display_xml
from MERci.analysis.stage_z   import read_off_file_if_ready, summarize_focus_lock
from MERci.acquisition.pipeline_config import load_pipeline_config

# PIPELINE_ID names this variant's own pipeline.yaml -- see notebook 01's own
# comment on PIPELINE_ID/PIPELINE_CONFIG for the rationale.
PIPELINE_ID     = "tumor_epi"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / f"{PIPELINE_ID}_pipeline.yaml")

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"

# SAMPLE_NAME is the TRUE top-level experiment id -- must match what notebooks
# 02/03 used, since the positions files and round_info rows below are named
# with it (see notebook 02's docstring for why this isn't SAMPLE_DIR.name).
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
# POSITIONS_TAG is what every positions_{...}.txt filename this notebook reads/builds
# actually uses -- SAMPLE_NAME alone in the flat layout, or SAMPLE_NAME_IMAGING_DIR in
# a split layout.
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────────
MICROSCOPE    = PIPELINE_CONFIG.microscope
# Focus-test movie length: 0 = check focus only, no movie. tumor_disk takes a
# 405 + blank movie per FOV, the input of 02d_filter_positions_by_tissue_quality.
N_TEST_FRAMES = PIPELINE_CONFIG.focus_test_frames

# Resolve the Kilroy config that will run this experiment (same as notebook 04).
KILROY_DIR    = MERCI_DIR / "data" / "configs" / "kilroy"
KILROY_CONFIG = find_kilroy_config(MICROSCOPE, KILROY_DIR, fallback_microscope="MF2")
print(f"MICROSCOPE    : {MICROSCOPE}")
print(f"N_TEST_FRAMES : {N_TEST_FRAMES}")
print(f"Kilroy config (protocol source): {KILROY_CONFIG.name}")

## Focus-lock test recipe

Runs FIRST, before notebook 04's cells/hybs recipes: builds a **separate** Dave recipe
that visits every FOV and checks focus lock only -- no fluidics -- to
catch a bad lock across the whole coverslip before committing to the
full multi-hour acquisition. Does not touch the cells/hybs recipes;
writes its own file.

`FOCUS_TEST_LEAD_MIN` (next cell) sets how many minutes from now this
recipe is assumed to start on the microscope (printed "Estimated
experiment cost" and the timing table at the end).

`N_TEST_FRAMES` comes from pipeline.yaml's `focus_test_frames`.

**`N_TEST_FRAMES = 0`: check-focus only, no image ever taken.**
A `<movie>` that omits `<length>`/`<parameters>` expands to a branch
containing ONLY `DAMoveStage` + `DACheckFocus` -- no `DASetParameters`/
`DATakeMovie`, no image taken, no HAL settings changed. No patch to
Dave/HAL is needed.

Trade-off: this mode leaves **no persisted per-FOV record** of whether the
lock was good. HAL's focus-status reply only reaches Dave live over TCP;
Dave shows a failed check in its own transient, in-memory warnings list
(never written to a file) -- watch that panel while the recipe runs.

**`N_TEST_FRAMES > 0`: also takes a real (short) movie per FOV.** Needs a
real HAL config/shutter pair -- generated by notebook 01's "create hal
config for focus test" section, alongside the other rounds' HAL configs, so
this notebook just resolves that file by name rather than building it
itself. This produces a genuine, persisted per-FOV record: HAL's normal
`.off` sidecar file (the same one `analysis/stage_z.py` already reads for
stage-z drift), whose `good-offset` column flags exactly which frame(s) had
a bad lock -- at the cost of real (small) disk usage/time per FOV. Read the
results back with the last cell below, after running the recipe on the
microscope. The same movies are what
`02d_filter_positions_by_tissue_quality.ipynb` scores.

**Positions:** the recipe visits `positions_{POSITIONS_TAG}_all.txt` when
02d has already run (02d keeps the full FOV list there and writes the kept
FOVs to `positions_{POSITIONS_TAG}.txt`), else `positions_{POSITIONS_TAG}.txt`.

In [ ]:
# ── Focus-lock test parameters ──────────────────────────────────────────
FOCUS_TEST_LEAD_MIN = 30    # minutes from now until this recipe is assumed to
                              # start on the microscope (see markdown)
NUM_FOCUS_CHECKS = 50
FOCUS_SCAN       = True

# Focus-test HAL config generated by notebook 01's "create hal config for
# focus test" section -- just resolve its filename here.
_focustest_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*-focustest-*.xml"))
if not _focustest_hal_configs:
    raise FileNotFoundError(
        f"No focus-test HAL config found in {SETTINGS_DIR} -- run notebook 01's "
        f"'create hal config for focus test' section first."
    )
TEST_HAL_CONFIG = _focustest_hal_configs[-1].name
print(f"Focus-test HAL config: {TEST_HAL_CONFIG}")

# tumor/ is single-tissue-per-coverslip, so notebook 02 always writes this aggregate
# FOV file (multiple boundaries within that one tissue don't change it). After
# 02d, the full list is the _all file (see markdown).
focus_test_positions = POSITIONS_DIR / f"positions_{POSITIONS_TAG}_all.txt"
if not focus_test_positions.exists():
    focus_test_positions = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
if not focus_test_positions.exists():
    raise FileNotFoundError(f"{focus_test_positions} not found -- run notebook 02 first.")

focus_test_output   = SETTINGS_DIR / dave_focustest_config_filename(MICROSCOPE, SAMPLE_NAME)
focus_test_data_dir = (SAMPLE_DIR / "data" / "focus_test") if N_TEST_FRAMES > 0 else None

focus_test_start = datetime.now() + timedelta(minutes=FOCUS_TEST_LEAD_MIN)

n_fovs, focustest_est = create_focus_test_dave_config(
    positions_file   = focus_test_positions,
    output_path      = focus_test_output,
    num_focus_checks = NUM_FOCUS_CHECKS,
    focus_scan       = FOCUS_SCAN,
    n_test_frames    = N_TEST_FRAMES,
    hal_config       = TEST_HAL_CONFIG,
    settings_dir     = SETTINGS_DIR,
    data_dir         = focus_test_data_dir,
    movie_name       = f"hal-{MICROSCOPE.lower()}-focustest",
    kilroy_config    = KILROY_CONFIG,
    microscope       = MICROSCOPE,
    start_time       = focus_test_start,
    per_round        = False,   # per-round breakdown shown as a table at the end instead
)

print(f"Focus-lock test recipe saved: {focus_test_output}")
print(f"Positions: {focus_test_positions.name}")
print(f"FOVs visited: {n_fovs}")
if N_TEST_FRAMES > 0:
    print(f"Each FOV takes a real {N_TEST_FRAMES}-frame movie -> .off files land under {focus_test_data_dir}")
else:
    print("Check-focus only -- no movie taken, no persisted per-FOV record (see markdown above).")

display_xml(focus_test_output)

In [ ]:
display(experiment_estimate_table(focustest_est, focus_test_start))

In [ ]:
# ── Read back per-FOV focus-lock results ──────────────────────────────────
# Only meaningful when N_TEST_FRAMES > 0 above AND the recipe has already been
# run on the microscope (each FOV's .off sidecar only exists once HAL has
# actually written that FOV's movie -- see the markdown above).
if N_TEST_FRAMES > 0:
    # Dave's own auto-increment padding (v2Generator's copyChildren) zero-pads
    # the FOV index to len(str(n_fovs)) digits -- confirmed directly against the
    # real source; distinct from MERci's own fov_pad_width convention used for
    # round_info.csv series patterns, so it is NOT reused here.
    pad = len(str(n_fovs))
    bad_fovs  = []
    n_checked = 0
    for fov_idx in range(n_fovs):
        off_path = focus_test_data_dir / f"hal-{MICROSCOPE.lower()}-focustest_{str(fov_idx).zfill(pad)}.off"
        # read_off_file_if_ready returns None both when the file doesn't exist
        # yet AND when HAL has created it but not finished writing it (a real
        # race when reading back while the recipe is still running on the
        # microscope) -- either way, just means "not ready yet, check again later".
        off_df = read_off_file_if_ready(off_path)
        if off_df is None:
            continue
        n_checked += 1
        summary = summarize_focus_lock(off_df)
        if not summary["all_good"]:
            bad_fovs.append((fov_idx, summary))

    print(f"Checked {n_checked} / {n_fovs} FOV(s) (missing ones haven't been imaged yet).")
    if bad_fovs:
        print(f"{len(bad_fovs)} FOV(s) with a bad focus lock:")
        for fov_idx, summary in bad_fovs:
            print(f"  FOV {fov_idx}: {summary['n_bad_frames']}/{summary['n_frames']} bad frame(s)")
    elif n_checked:
        print("All checked FOVs had a good focus lock.")
    else:
        print("No FOVs found yet -- run the recipe on the microscope first.")
else:
    print("N_TEST_FRAMES == 0 -- no per-FOV file was written (see markdown above); "
          "check Dave's own warnings panel while the recipe runs instead.")